In [1]:
# from IPython.display import IFrame
# from docling.document_converter import DocumentConverter
# import boto3
# import os
# from sdg_hub.core.flow import FlowRegistry
# from sdg_hub.core.blocks import BlockRegistry
# import pypdfium2 as pdfium
# from langchain_openai import ChatOpenAI
# from langchain_community.vectorstores import LanceDB
# from langchain_community.embeddings import OpenAIEmbeddings
# from langchain_community.document_loaders import TextLoader
# from langchain_community.graph_vectorstores import GraphVectorStoreRetriever
# from langchain_core.documents import Document
# from lancedb.rerankers import LinearCombinationReranker
# from langchain_openai import OpenAIEmbeddings
# from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
# from langchain.docstore.document import Document
# from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
# import lancedb
# from huggingface_hub import snapshot_download
# from langchain_community.embeddings import HuggingFaceBgeEmbeddings, SentenceTransformerEmbeddings
# from transformers import AutoTokenizer
# from enum import Enum
# import traceback
# from sdg_hub import Flow, FlowRegistry
# from dotenv import load_dotenv
# import re

In [2]:
# load_dotenv()

In [3]:
# endpoint_url = os.getenv('AWS_S3_ENDPOINT')
# access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
# secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
# config = boto3.session.Config(signature_version='s3v4')
# bucket = os.getenv("AWS_S3_BUCKET")
# # source_path = 'pdf/'
# # target_path = 'pdf'
# # target_path_chapters = 'pdf_chunked'
# # target_path_markdown = 'markdown'
# source_path = 'pdf_source/'
# target_path = 'pdf_target'
# target_path_chapters = 'pdf_chunked_target'
# target_path_markdown = 'markdown_target'
# CODE_LANGUAGE='ColdFusion'


# embedding_model = SentenceTransformerEmbeddings(
#     model_name="BAAI/bge-small-en-v1.5", 
#     model_kwargs={"trust_remote_code":True
# })

# llm = ChatOpenAI(
#     model="openai/gpt-oss-20b", # os.getenv('QWEN25CODER_MODEL_ID'),
#     api_key=os.getenv('OPENROUTER_TOKEN'),
#     base_url=os.getenv('OPENROUTER_API_BASE'),
#     temperature=0.1,
# )

# vectorstore_connection = lancedb.connect(f"s3://data/lancedb-graphrag",
#     storage_options={
#         "endpoint_url": endpoint_url,
#         "aws_access_key_id": access_key_id,
#         "aws_secret_access_key": secret_access_key,
#         "s3_force_path_style": "true",
#         "allow_http": "true",
#     }
# )

# vectorstore = LanceDB(
#     mode="append",
#     embedding=embedding_model,
#     connection=vectorstore_connection,
# )

# minio = boto3.client(
#     's3',
#     endpoint_url=endpoint_url,
#     aws_access_key_id=access_key_id,
#     aws_secret_access_key=secret_access_key,
#     config=boto3.session.Config(signature_version='s3v4')
# )

In [4]:
def get_processable_files(src):
    """
    Returns a list of processable files from the given path.
    """
    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    
    files = [f for f in os.listdir(src) if ".pdf" in f]

    return files

In [5]:
def get_chapter_ranges(sourcefilename, do_print=True):
    """
    Returns a list of (beginPage, endPage) ranges for chunks that represent chapters in the given pdf.
    """

    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    import pypdfium2 as pdfium
    
    print("Getting chapter ranges...\n")
    
    pdf = pdfium.PdfDocument(sourcefilename)
    
    ranges = []
    
    begin, end = None, None
    
    for item in pdf.get_toc():
        
        state = "*" if item.n_kids == 0 else "-" if item.is_closed else "+"
        
        target = "?" if item.page_index is None else item.page_index+1
        
        boundary = None
        
        if item.page_index and ((item.n_kids == 0 and item.level < 2) or item.level == 2):
            
            if begin is not None:
                
                end = item.page_index - 1
                
                boundary = [begin, max(begin, end)]
                
                ranges.append(boundary)
                
            begin = item.page_index
            
        if do_print:
            
            if boundary:
                
                print("    " * 2 +  f"(Pages {(boundary[0]+1)} - {(boundary[1]+1)})" + "\n")
                
            print(("    " * item.level) + f"[{state}] {item.title} -> {target}  # {item.view_mode} {item.view_pos}")
            
    return ranges

In [6]:
def split_chapters(sourcefilename, targetfilename, pagerange):
    """
    Splits the pdf into chapters using the provided page ranges.
    Returns the name of the new pdf chunk.
    """

    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    import pypdfium2 as pdfium
    from pathlib import Path
    
    try:
        
        source_pdf = pdfium.PdfDocument(sourcefilename)
        
        new_pdf = pdfium.PdfDocument.new()
    
        print(f"Retrieving chapter...{targetfilename}, Pages {pagerange[0]} to {pagerange[1]}")
        
        new_page_index = new_pdf.import_pages(source_pdf, pages=list(range(pagerange[0], pagerange[1]+1)))
        
        new_pdf.save(targetfilename)
        
        source_pdf.close()
        
        new_pdf.close()
        
    except Exception as e:
        
        print(f"Error saving {targetfilename}: {e}")

In [7]:
def convert_to_markdown(pdffile, markdownfile):
    """
    Converts the pdf into a markdown file.
    """
    
    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    from docling.document_converter import DocumentConverter
    
    try:
        print(f"Converting {pdffile} to markdown...")
        
        converter = DocumentConverter()
        
        result = converter.convert(pdffile)
        
        markdown_output = result.document.export_to_markdown()

        with open(markdownfile, "w") as file:
            
            file.write(markdown_output)

        print(f"{markdownfile} generated.")
        
    except Exception as e:
        print(f"Error saving {markdownfile}: {e}")
    

In [8]:
def generate_markdown_section_raw_data(file):
    """
    Generates markdown section chunks from the file.
    """

    ##############################################
    # Imports
    ##############################################
    from datasets import Dataset, Features, Value
    from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
    from langchain.docstore.document import Document
    from sdg_hub.core.blocks import PromptBuilderBlock, LLMChatBlock, LLMParserBlock
    from datasets import Dataset, concatenate_datasets
    import traceback
    import re

    dataset = None

    def strip_code_section(content):
        """
        Strips out code sections of file.
        """
        code_sections = re.findall(r'```([^`]+)```', content, re.DOTALL)
        return code_sections
    
    try:
        print(f"Parsing markdown {file}...")
        
        filecontent = None
        
        with open(file, mode="r") as f: 
            
            filecontent = f.read()

            if strip_code_section(filecontent):

                print(f"Starting code-to-text mappings for {file}...")
                
                headers_to_split = [("#", "Header 1"), ("##", "Header 2"),("###", "Header 3")]
                
                text_splitter = MarkdownHeaderTextSplitter(headers_to_split, strip_headers=False)
            
                splits = text_splitter.split_text(filecontent)
        
                sections = [strip_code_section(split.page_content) for split in splits]
    
                
                dataset = Dataset.from_dict({
                   "code":section for section in sections if section
                })

                dataset.map(lambda x: dict(code_summary="",code_components="",
                                             code_domain="",code_topics="",
                                             evaluation_code_summary_faithfulness="",
                                             evaluation_code_summary_relevance="",
                                             evaluation_code_components_faithfulness="",
                                             evaluation_code_components_relevance="",
                                             evaluation_code_topics_faithfulness="",
                                             evaluation_code_topics_relevance=""))

        return dataset        

    except Exception as e:

        print(f"Error occurred while parsing markdown {file}: {e}")

        traceback.print_exc()

In [9]:
##############################################
# Imports
##############################################
from dotenv import load_dotenv
import os
load_dotenv()
from pathlib import Path
from datasets import Dataset, concatenate_datasets

source_path = 'pdf_source'

target_path_chapters = 'pdf_chunked_target'

target_path_markdown = 'pdf_chunked_markdown'

target_path_jsonl = "json"

for directory_path in [source_path, 
                       
                       target_path_chapters, 
                       
                       target_path_markdown,
                      
                       target_path_jsonl]:
        
    Path(directory_path).mkdir(parents=True, exist_ok=True)

files = get_processable_files(source_path)

dataset = None

for file in files:
    
    ranges = get_chapter_ranges(f"{source_path}/{file}", do_print=False)
    
    for idx, _range in enumerate(ranges):
        
        pdf = f"{target_path_chapters}/{idx}_{file}"
        
        md = f"{target_path_markdown}/{idx}_{file.replace('.pdf', '.md')}"
        
        # split_chapters(f"{source_path}/{file}", pdf, _range)
        
        # convert_to_markdown(pdf, md)

        dataset = generate_markdown_section_raw_data(md) if not dataset else concatenate_datasets([dataset, generate_markdown_section_raw_data(md)])

print("Writing dataset to jsonl file...")

dataset.to_json(f"{target_path_jsonl}/data.jsonl")

Getting chapter ranges...

Parsing markdown pdf_chunked_markdown/0_Developing_Apps_coldfusion.md...
Parsing markdown pdf_chunked_markdown/1_Developing_Apps_coldfusion.md...
Parsing markdown pdf_chunked_markdown/2_Developing_Apps_coldfusion.md...
Parsing markdown pdf_chunked_markdown/3_Developing_Apps_coldfusion.md...
Parsing markdown pdf_chunked_markdown/4_Developing_Apps_coldfusion.md...
Parsing markdown pdf_chunked_markdown/5_Developing_Apps_coldfusion.md...
Parsing markdown pdf_chunked_markdown/6_Developing_Apps_coldfusion.md...
Starting code-to-text mappings for pdf_chunked_markdown/6_Developing_Apps_coldfusion.md...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Parsing markdown pdf_chunked_markdown/7_Developing_Apps_coldfusion.md...
Starting code-to-text mappings for pdf_chunked_markdown/7_Developing_Apps_coldfusion.md...


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Parsing markdown pdf_chunked_markdown/8_Developing_Apps_coldfusion.md...
Starting code-to-text mappings for pdf_chunked_markdown/8_Developing_Apps_coldfusion.md...


...

Parsing markdown pdf_chunked_markdown/41_Developing_Apps_coldfusion.md...
Starting code-to-text mappings for pdf_chunked_markdown/41_Developing_Apps_coldfusion.md...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Writing dataset to jsonl file...


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

28389

In [10]:
# try:
#     os.makedirs(target_path, exist_ok=True)
#     os.makedirs(target_path_chapters, exist_ok=True)
#     files = minio.list_objects_v2(Bucket=bucket, Prefix=source_path)
#     if 'Contents' in files:
#         for obj in files['Contents']:
#             file = obj['Key']
#             minio.download_file(bucket, file, f"{target_path}/{file.split('/')[-1]}")
#             print(f"File '{source_path}' downloaded successfully to {target_path}/{file.split('/')[-1]}")
# except Exception as e:
#     print(f"Error downloading file: {e}")

In [11]:
# table = vectorstore_connection.open_table('vectorstore')
# table_schema = table.schema
# print(f"Schema for table '{table.name}':")
# print("-" * 30)
# for field in table_schema:
#     print(f" - Column: '{field.name}'")
#     print(f"   Type: {field.type}")
#     print(f"   Nullable: {field.nullable}")

# print(f"\nFull PyArrow Schema:\n{table_schema}")
